In [ ]:
import traceback
import pandas as pd
import yfinance as yf

TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS"]  # edit as needed

In [ ]:
def inspect_fields(ticker: str = "RELIANCE.NS") -> None:
    t = yf.Ticker(ticker)
    print(f"--- {ticker} quarterly_financials row labels ---")
    print(list(t.quarterly_financials.index))
    print(f"\n--- {ticker} quarterly_balance_sheet row labels ---")
    print(list(t.quarterly_balance_sheet.index))
    print(f"\n--- {ticker} quarterly_cashflow row labels ---")
    print(list(t.quarterly_cashflow.index))
    print(f"\n--- {ticker} quarter-end columns available ---")
    print(list(t.quarterly_financials.columns))

# inspect_fields("RELIANCE.NS")

In [ ]:
FIELD_CANDIDATES = {
    "net_income": ["Net Income", "Net Income Common Stockholders"],
    "ebit": ["EBIT", "Operating Income"],
    "total_equity": ["Stockholders Equity", "Total Stockholder Equity",
                      "Common Stock Equity"],
    "total_assets": ["Total Assets"],
    "current_liabilities": ["Current Liabilities",
                             "Total Current Liabilities"],
    "total_debt": ["Total Debt"],
    "operating_cash_flow": ["Operating Cash Flow",
                             "Total Cash From Operating Activities"],
}


def _get_row(df: pd.DataFrame, candidates: list[str]):
    for label in candidates:
        if label in df.index:
            return df.loc[label]
    return None


def _strip_tz(idx):
    """Make a DatetimeIndex timezone-naive if it isn't already —
    this is what fixes the comparison error you hit."""
    idx = pd.to_datetime(idx)
    return idx.tz_localize(None) if idx.tz is not None else idx

In [ ]:
def collect_ticker(ticker: str) -> pd.DataFrame:
    t = yf.Ticker(ticker)

    qf = t.quarterly_financials
    qbs = t.quarterly_balance_sheet
    qcf = t.quarterly_cashflow

    net_income = _get_row(qf, FIELD_CANDIDATES["net_income"])
    ebit = _get_row(qf, FIELD_CANDIDATES["ebit"])
    total_equity = _get_row(qbs, FIELD_CANDIDATES["total_equity"])
    total_assets = _get_row(qbs, FIELD_CANDIDATES["total_assets"])
    current_liab = _get_row(qbs, FIELD_CANDIDATES["current_liabilities"])
    total_debt = _get_row(qbs, FIELD_CANDIDATES["total_debt"])
    op_cash_flow = _get_row(qcf, FIELD_CANDIDATES["operating_cash_flow"])

    quarter_ends = _strip_tz(qf.columns)  # <-- tz stripped here

    rows = []
    for q_end in sorted(quarter_ends):
        rows.append({
            "ticker": ticker,
            "quarter_end": q_end,
            "net_income": net_income.get(q_end) if net_income is not None else None,
            "ebit": ebit.get(q_end) if ebit is not None else None,
            "total_equity": total_equity.get(q_end) if total_equity is not None else None,
            "total_assets": total_assets.get(q_end) if total_assets is not None else None,
            "current_liabilities": current_liab.get(q_end) if current_liab is not None else None,
            "total_debt": total_debt.get(q_end) if total_debt is not None else None,
            "operating_cash_flow": op_cash_flow.get(q_end) if op_cash_flow is not None else None,
        })

    raw = pd.DataFrame(rows).sort_values("quarter_end").reset_index(drop=True)

    raw["ROE"] = raw["net_income"] / raw["total_equity"]
    raw["ROCE"] = raw["ebit"] / (raw["total_assets"] - raw["current_liabilities"])
    raw["Debt_Equity"] = raw["total_debt"] / raw["total_equity"]
    raw["Cash_Flow_Generation"] = raw["operating_cash_flow"]
    raw["Profit_Growth_QoQ"] = raw["net_income"].pct_change(periods=1)
    raw["Profit_Growth_YoY"] = raw["net_income"].pct_change(periods=4)

    try:
        earnings = t.get_earnings_dates(limit=40)
        earnings_dates = _strip_tz(earnings.index).sort_values()  # <-- tz stripped here
    except Exception:
        earnings_dates = pd.DatetimeIndex([])

    def nearest_earnings_date(q_end):
        if len(earnings_dates) == 0:
            return pd.NaT
        candidates = earnings_dates[earnings_dates >= q_end]
        return candidates.min() if len(candidates) else pd.NaT

    raw["earnings_release_date"] = raw["quarter_end"].apply(nearest_earnings_date)

    keep_cols = ["ticker", "quarter_end", "earnings_release_date",
                 "ROE", "ROCE", "Debt_Equity", "Cash_Flow_Generation",
                 "Profit_Growth_QoQ", "Profit_Growth_YoY"]
    return raw[keep_cols]

In [ ]:
def collect_all(tickers: list[str]) -> pd.DataFrame:
    frames = []
    for tk in tickers:
        print(f"Pulling {tk} ...")
        try:
            frames.append(collect_ticker(tk))
        except Exception:
            print(f"  Failed for {tk}:")
            traceback.print_exc()

    combined = pd.concat(frames, ignore_index=True)
    print("\n--- Missing value summary (nothing has been filled) ---")
    print(combined.isna().sum())
    return combined

In [ ]:
fundamentals_df = collect_all(TICKERS)
fundamentals_df.to_csv("fundamentals_raw.csv", index=False)
print("\nSaved fundamentals_raw.csv")
fundamentals_df

In [36]:
import os, time
import numpy as np
import pandas as pd
import yfinance as yf

UNIVERSE = [
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "ITC.NS", "HINDUNILVR.NS", "LT.NS",
    "MARUTI.NS", "SUNPHARMA.NS", "ASIANPAINT.NS", "TITAN.NS", "ULTRACEMCO.NS",
    "NESTLEIND.NS", "WIPRO.NS", "HCLTECH.NS", "ONGC.NS", "NTPC.NS", "BAJAJ-AUTO.NS",
    "TATASTEEL.NS", "POWERGRID.NS", "COALINDIA.NS", "M&M.NS", "TECHM.NS",
    "DRREDDY.NS", "CIPLA.NS", "BRITANNIA.NS", "HEROMOTOCO.NS", "EICHERMOT.NS",
    "GRASIM.NS", "JSWSTEEL.NS", "HINDALCO.NS",
]   # non-financials only (banks/NBFCs need different ratios)

RAW_STORE = "fundamentals_store.csv"
DATE_COLS = ["quarter_end", "bs_date", "release_date_ist"]
PRICE_START = "2021-01-01"
FEATURES = ["ROE", "ROCE", "Debt_Equity", "OCF_to_Assets",
            "Profit_Growth_QoQ", "Profit_Growth_YoY"]

In [40]:
def collect_raw(ticker):
    """Raw levels per quarter; balance-sheet items and annual OCF are as-of the latest
    statement dated <= quarter_end."""
    t = yf.Ticker(ticker)
    qf, qbs, qcf, abs_, acf = (t.quarterly_financials, t.quarterly_balance_sheet,
                               t.quarterly_cashflow, t.balance_sheet, t.cashflow)
    if qf is None or qf.empty:
        raise ValueError("no quarterly financials returned")

    q = pd.DataFrame({"quarter_end": sorted(set(_naive(qf.columns)))})
    q["net_income"] = q["quarter_end"].map(_ser(qf, LABELS["net_income"]))
    q["ebit"] = q["quarter_end"].map(_ser(qf, LABELS["ebit"]))
    q["ocf"] = q["quarter_end"].map(_ser(qcf, LABELS["ocf"]))

    bs = pd.DataFrame({k: _ser(qbs, LABELS[k]).combine_first(_ser(abs_, LABELS[k]))
                       for k in ["equity", "assets", "cur_liab", "debt"]}).sort_index()
    if len(bs):
        bs = bs.ffill()
        bs.index.name = "bs_date"
        q = pd.merge_asof(q, bs.reset_index().astype({"bs_date": "datetime64[ns]"}),
                          left_on="quarter_end", right_on="bs_date", direction="backward")
    else:
        for c in ["equity", "assets", "cur_liab", "debt", "bs_date"]:
            q[c] = np.nan

    # annual operating cash flow, as-of the latest fiscal year dated <= quarter_end
    ocf_a = _ser(acf, LABELS["ocf"])
    if len(ocf_a):
        a = pd.DataFrame({"ocf_annual": ocf_a}).rename_axis("cf_date").reset_index()
        a["cf_date"] = a["cf_date"].astype("datetime64[ns]")
        q = pd.merge_asof(q, a, left_on="quarter_end", right_on="cf_date", direction="backward")
        q = q.drop(columns="cf_date")
    else:
        q["ocf_annual"] = np.nan

    ed = get_earnings_dates_ist(ticker)
    def pick(qe):
        if len(ed):
            c = ed[(ed >= qe + pd.Timedelta(days=1)) & (ed <= qe + pd.Timedelta(days=75))]
            if len(c):
                return c.min()
        return pd.NaT
    q["release_date_ist"] = q["quarter_end"].apply(pick)
    q.insert(0, "ticker", ticker)
    return q

In [41]:
raw_new = collect_universe(UNIVERSE)
store = update_store(raw_new)

print(f"\nStore: {len(store)} rows, {store['ticker'].nunique()} tickers -> {RAW_STORE}")
print(store.groupby("ticker").agg(quarters=("quarter_end", "size"),
                                  first_q=("quarter_end", "min"),
                                  last_q=("quarter_end", "max")))
print("\nNon-null counts:\n", store[["net_income", "ebit", "equity", "assets",
                                     "cur_liab", "debt", "ocf"]].notna().sum())

Pulling RELIANCE.NS ...
Pulling TCS.NS ...
Pulling INFY.NS ...
Pulling ITC.NS ...
Pulling HINDUNILVR.NS ...
Pulling LT.NS ...
Pulling MARUTI.NS ...
Pulling SUNPHARMA.NS ...
Pulling ASIANPAINT.NS ...
Pulling TITAN.NS ...
Pulling ULTRACEMCO.NS ...
Pulling NESTLEIND.NS ...
Pulling WIPRO.NS ...
Pulling HCLTECH.NS ...
Pulling ONGC.NS ...
Pulling NTPC.NS ...
Pulling BAJAJ-AUTO.NS ...
Pulling TATASTEEL.NS ...
Pulling POWERGRID.NS ...
Pulling COALINDIA.NS ...
Pulling M&M.NS ...
Pulling TECHM.NS ...
Pulling DRREDDY.NS ...
Pulling CIPLA.NS ...
Pulling BRITANNIA.NS ...
Pulling HEROMOTOCO.NS ...
Pulling EICHERMOT.NS ...
Pulling GRASIM.NS ...
Pulling JSWSTEEL.NS ...
Pulling HINDALCO.NS ...

Store: 170 rows, 30 tickers -> fundamentals_store.csv
               quarters    first_q     last_q
ticker                                       
ASIANPAINT.NS         5 2025-06-30 2026-06-30
BAJAJ-AUTO.NS         5 2025-06-30 2026-06-30
BRITANNIA.NS          6 2024-12-31 2026-06-30
CIPLA.NS              6 2024-

In [48]:
def compute_features(store):
    m = store.copy()
    for c in DATE_COLS:
        m[c] = pd.to_datetime(m[c]).astype("datetime64[ns]")
    m = m.sort_values(["ticker", "quarter_end"]).reset_index(drop=True)

    m["ROE"] = m["net_income"] / m["equity"]
    m["ROCE"] = m["ebit"] / (m["assets"] - m["cur_liab"])
    m["Debt_Equity"] = m["debt"] / m["equity"]

    r = m["ocf_annual"] / m["assets"]                  # annual OCF / total assets
    bad = r.abs().lt(1e-3) | r.abs().gt(0.5)           # implausible -> unit/currency mismatch
    if bad.any():
        print("WARNING: OCF_to_Assets set to NaN (unit/currency mismatch?) for:",
              sorted(m.loc[bad, "ticker"].unique()))
    m["OCF_to_Assets"] = r.where(~bad)

    for name in ["Profit_Growth_QoQ", "Profit_Growth_YoY"]:
        m[name] = np.nan
    for tk, g in m.groupby("ticker"):
        per = g["quarter_end"].dt.to_period("Q")
        s = pd.Series(g["net_income"].values, index=per)
        for name, lag in [("Profit_Growth_QoQ", 1), ("Profit_Growth_YoY", 4)]:
            prev = s.reindex(per - lag).values
            m.loc[g.index, name] = np.clip((g["net_income"].values - prev) / np.abs(prev), -1, 1)

    dl = m["quarter_end"] + pd.to_timedelta(np.where(m["quarter_end"].dt.month == 3, 60, 45), unit="D")
    m["release_source"] = np.where(m["release_date_ist"].notna(), "actual", "deadline")
    m["release_date_ist"] = m["release_date_ist"].fillna(dl)

    return m[["ticker", "quarter_end", "release_date_ist", "release_source",
              "ROE", "ROCE", "Debt_Equity", "OCF_to_Assets",
              "Profit_Growth_QoQ", "Profit_Growth_YoY"]]

print("uses ocf_annual:", 'm["ocf_annual"]' in __import__("inspect").getsource(compute_features))

uses ocf_annual: True


In [50]:
fund_long = compute_features(store)
print("OCF_to_Assets available:", fund_long["OCF_to_Assets"].notna().sum(), "of", len(fund_long))

fund = prepare_fund(fund_long, FEATURES)
tickers = sorted(fund["ticker"].unique())
days = get_trading_days(tickers, PRICE_START)
daily = align_to_daily(fund, days)

LEVEL_COLS = ["ROE", "ROCE", "Debt_Equity"]
GROWTH_COLS = ["Profit_Growth_QoQ", "Profit_Growth_YoY"]

clean = daily.copy()
for c in GROWTH_COLS:
    clean[c] = clean[c].fillna(0.0).clip(-1, 1)

# Missing OCF -> ONE constant (not date-varying); *_stale flag marks these rows
ocf_const = fund_long["OCF_to_Assets"].median()
if pd.isna(ocf_const):
    ocf_const = 0.0
clean["OCF_to_Assets"] = clean["OCF_to_Assets"].fillna(ocf_const)

clean = clean.dropna(subset=LEVEL_COLS + ["days_since_earnings"]).reset_index(drop=True)

cols = (["ticker", "date", "days_since_earnings"] + FEATURES
        + [f"{c}_stale" for c in FEATURES])
clean = clean[cols]
assert clean.isna().sum().sum() == 0, "nulls remain"

core = [c for c in FEATURES if c != "OCF_to_Assets"]
print("\nTickers kept:", clean["ticker"].nunique(), "of", len(UNIVERSE))
print("Distinct vectors excl. OCF:", len(clean[core].drop_duplicates()))
print("Distinct vectors incl. OCF:", len(clean[FEATURES].drop_duplicates()))
print("Rows with imputed OCF     :", int((clean["OCF_to_Assets_stale"] == 1).sum()), "of", len(clean))

clean.to_csv("fundamentals_daily_clean_v2.csv", index=False)

OCF_to_Assets available: 170 of 170

Tickers kept: 30 of 30
Distinct vectors excl. OCF: 150
Distinct vectors incl. OCF: 150
Rows with imputed OCF     : 0 of 9769


In [47]:
import inspect
src = inspect.getsource(compute_features)
print("compute_features uses ocf_annual:", 'm["ocf_annual"]' in src)

r = store["ocf_annual"] / store["assets"]
print("\nannual OCF / assets across the store:")
print(r.describe())
print("Rows the plausibility guard would blank:", int((r.abs().lt(1e-3) | r.abs().gt(0.5)).sum()), "of", len(r))

compute_features uses ocf_annual: False

annual OCF / assets across the store:
count    170.000000
mean       0.128575
std        0.079416
min       -0.034303
25%        0.088491
50%        0.125684
75%        0.152876
max        0.306385
dtype: float64
Rows the plausibility guard would blank: 0 of 170


In [43]:
print("ocf_annual" in store.columns)
print(store[["net_income", "ebit", "equity", "assets", "cur_liab", "debt",
             "ocf", "ocf_annual"]].notna().sum())

True
net_income    150
ebit          150
equity        170
assets        170
cur_liab      170
debt          170
ocf            20
ocf_annual    170
dtype: int64


In [45]:
core = [c for c in FEATURES if c != "OCF_to_Assets"]
print("Distinct vectors excl. OCF :", len(clean[core].drop_duplicates()))
print("Distinct vectors incl. OCF :", len(clean[FEATURES].drop_duplicates()))
print("Tickers with imputed OCF   :",
      sorted(clean.loc[clean["OCF_to_Assets_stale"] == 1, "ticker"].unique()))

Distinct vectors excl. OCF : 150
Distinct vectors incl. OCF : 457
Tickers with imputed OCF   : ['ASIANPAINT.NS', 'BAJAJ-AUTO.NS', 'BRITANNIA.NS', 'CIPLA.NS', 'COALINDIA.NS', 'EICHERMOT.NS', 'GRASIM.NS', 'HCLTECH.NS', 'HEROMOTOCO.NS', 'HINDALCO.NS', 'HINDUNILVR.NS', 'INFY.NS', 'ITC.NS', 'JSWSTEEL.NS', 'LT.NS', 'M&M.NS', 'MARUTI.NS', 'NESTLEIND.NS', 'NTPC.NS', 'ONGC.NS', 'POWERGRID.NS', 'RELIANCE.NS', 'SUNPHARMA.NS', 'TATASTEEL.NS', 'TCS.NS', 'TECHM.NS', 'TITAN.NS', 'ULTRACEMCO.NS']


In [51]:
fl = compute_features(store)

print("Store rows missing net_income or ebit (-> ROE/ROCE NaN):")
miss = store[store["net_income"].isna() | store["ebit"].isna()]
print(miss[["ticker", "quarter_end", "net_income", "ebit", "equity", "assets", "cur_liab"]].to_string())

print("\nFeatures table  - NaN :", fl[["ROE", "ROCE"]].isna().sum().to_dict())
print("Features table  - zero:", (fl[["ROE", "ROCE"]] == 0).sum().to_dict())
print("Final clean file - NaN :", clean[["ROE", "ROCE"]].isna().sum().to_dict())
print("Final clean file - zero:", (clean[["ROE", "ROCE"]] == 0).sum().to_dict())
print("\nFinal clean file summary:")
print(clean[["ROE", "ROCE"]].describe().round(4))

Store rows missing net_income or ebit (-> ROE/ROCE NaN):
            ticker quarter_end  net_income  ebit        equity        assets      cur_liab
10    BRITANNIA.NS  2024-12-31         NaN   NaN  3.941520e+10  9.073560e+10  4.084410e+10
16        CIPLA.NS  2024-12-31         NaN   NaN  2.670643e+11  3.271782e+11  5.245770e+10
27      DRREDDY.NS  2024-12-31         NaN   NaN  2.805500e+11  3.875180e+11  9.603900e+10
28      DRREDDY.NS  2025-03-31         NaN   NaN  3.333880e+11  4.929890e+11  1.303920e+11
34    EICHERMOT.NS  2024-12-31         NaN   NaN  1.804553e+11  2.312793e+11  3.598410e+10
35    EICHERMOT.NS  2025-03-31         NaN   NaN  2.129649e+11  2.717441e+11  4.265780e+10
41       GRASIM.NS  2024-12-31         NaN   NaN  8.865240e+11  4.125391e+12  9.243284e+11
49      HCLTECH.NS  2025-09-30         NaN   NaN  6.965500e+11  1.055440e+12  2.803900e+11
53   HEROMOTOCO.NS  2025-03-31         NaN   NaN  1.927204e+11  2.839032e+11  7.475980e+10
59     HINDALCO.NS  2025-03-31   

In [52]:
chk = store[store["ticker"] == "INFY.NS"][["quarter_end", "bs_date", "net_income", "equity", "assets"]]
print(chk.to_string())

fl = compute_features(store)
print("\nINFY ROE / ROCE by quarter:")
print(fl[fl["ticker"] == "INFY.NS"][["quarter_end", "ROE", "ROCE"]].round(4).to_string())

   quarter_end    bs_date   net_income        equity        assets
71  2024-12-31 2024-03-31          NaN  1.055900e+10  1.652300e+10
72  2025-03-31 2025-03-31          NaN  1.120500e+10  1.741900e+10
73  2025-06-30 2025-06-30  809000000.0  1.106900e+10  1.744700e+10
74  2025-09-30 2025-09-30  839000000.0  1.163400e+10  1.806400e+10
75  2025-12-31 2025-12-31  747000000.0  9.233000e+09  1.595300e+10
76  2026-03-31 2026-03-31  919000000.0  9.786000e+09  1.644600e+10
77  2026-06-30 2026-06-30  819000000.0  9.619000e+09  1.635900e+10

INFY ROE / ROCE by quarter:
   quarter_end     ROE    ROCE
71  2024-12-31     NaN     NaN
72  2025-03-31     NaN     NaN
73  2025-06-30  0.0731  0.0935
74  2025-09-30  0.0721  0.0917
75  2025-12-31  0.0809  0.1006
76  2026-03-31  0.0939  0.1080
77  2026-06-30  0.0851  0.1092


C:\Users\Manshi Rani\AppData\Local\Temp\ipykernel_33396\2856571488.py:6: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(fl[fl["ticker"] == "INFY.NS"][["quarter_end", "ROE", "ROCE"]].round(4).to_string())
